# 🔎 Análise Profunda das Anomalias Fiscais
## Cadeia de Combustíveis RJ — Casos Críticos e Padrões de Risco

**TCC MBA Data Science | USP ESALQ**  
**Aluna:** Kátia Rios Nóbrega de Mello  

---

### Objetivo
Analisar em profundidade os casos detectados como anômalos pelos modelos
Isolation Forest e DBSCAN, traduzindo os resultados para a linguagem
da fiscalização tributária e identificando padrões de risco acionáveis
para auditores fiscais.

### Estrutura
1. Configuração e carregamento
2. Visão geral dos resultados da modelagem
3. Análise dos 8 casos críticos
4. Análise dos clusters suspeitos (DBSCAN)
5. Padrões de anomalia identificados
6. Implicações para a fiscalização fiscal
7. Protocolo de priorização para auditores

## 1. Configuração e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

AZUL = {
    'escuro':  '#003366',
    'medio':   '#1a5276',
    'claro':   '#2e86c1',
    'palido':  '#85c1e9',
    'cinza':   '#566573',
}
VERMELHO = '#C0392B'

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlecolor'] = AZUL['escuro']
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

OUT_DIR = Path('../outputs/graficos')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Configuração carregada')

In [ ]:
# Carregar datasets
resultado = pd.read_parquet('../data/processed/resultado_modelagem.parquet')
features  = pd.read_parquet('../data/processed/dataset_features_rj.parquet')

# Juntar resultado com features
df = resultado.merge(
    features[[
        'id_empresa', 'anos_operacao', 'idade_empresa_dias',
        'qtd_socios', 'qtd_socios_pj', 'qtd_socios_pf',
        'qtd_filiais', 'capital_social', 'max_empresas_por_socio',
        'sem_socios', 'inapta', 'suspensa', 'taxa_inapta_municipio',
        'socio_risco_alto', 'capital_muito_baixo', 'capital_adequado',
        'antiga_inapta', 'empresa_muito_nova', 'postos_no_municipio',
        'score_risco_v2', 'classe_risco'
    ]],
    on='id_empresa', how='left'
)

print(f'Dataset combinado: {len(df):,} empresas, {df.shape[1]} variáveis')
print(f'Distribuição classe final:')
print(df['classe_final'].value_counts().sort_index())

## 2. Visão Geral dos Resultados da Modelagem

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribuição score final
axes[0].hist(df['score_final'], bins=30, color=AZUL['claro'],
              edgecolor='white', linewidth=1.2, alpha=0.85)
axes[0].axvline(0.70, color=VERMELHO, linestyle='--', linewidth=2,
                label='Limiar crítico (0.70)')
axes[0].axvline(0.50, color=AZUL['escuro'], linestyle='--', linewidth=1.5,
                label='Limiar alto (0.50)')
axes[0].set_title('Distribuição do Score Final Combinado')
axes[0].set_xlabel('Score Final'); axes[0].set_ylabel('Empresas')
axes[0].legend(fontsize=9)

# IF vs DBSCAN — concordância
tab = pd.crosstab(df['if_anomalia'], df['dbscan_outlier'],
                   rownames=['IF Anômalo'], colnames=['DBSCAN Outlier'])
sns.heatmap(tab, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            linewidths=0.5, annot_kws={'size':12,'fontweight':'bold'})
axes[1].set_title('Concordância IF vs DBSCAN\n(número de empresas)')

# Score final por situação cadastral
ordem = ['Ativa','Suspensa','Inapta']
dados = [df[df['situacao_desc']==s]['score_final'].values for s in ordem]
bp = axes[2].boxplot(dados, tick_labels=ordem, patch_artist=True,
                      medianprops={'color':VERMELHO,'linewidth':2.5})
cores = [AZUL['claro'], AZUL['medio'], AZUL['escuro']]
for patch, cor in zip(bp['boxes'], cores):
    patch.set_facecolor(cor); patch.set_alpha(0.85)
axes[2].set_title('Score Final por Situação Cadastral')
axes[2].set_ylabel('Score Final')

plt.suptitle('Visão Geral dos Resultados — Cadeia de Combustíveis RJ',
             fontsize=13, fontweight='bold', color=AZUL['escuro'])
plt.tight_layout()
plt.savefig(OUT_DIR/'11_visao_geral_resultados.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

## 3. Análise dos 8 Casos Críticos

In [ ]:
criticos = df[df['classe_final']=='Crítico'].sort_values('score_final', ascending=False).copy()

# Função para categorizar capital sem expor valores exatos
def faixa_capital(v):
    if v > 1e9:   return 'Acima de R$ 1 bilhão'
    if v > 1e6:   return 'Entre R$ 1M e R$ 1 bilhão'
    if v > 1e5:   return 'Entre R$ 100k e R$ 1M'
    if v > 1e4:   return 'Entre R$ 10k e R$ 100k'
    if v > 0:     return 'Abaixo de R$ 10k'
    return 'Zero'

criticos['faixa_capital'] = criticos['capital_social'].apply(faixa_capital)

# Tabela resumo dos 8 críticos
resumo = criticos[[
    'id_empresa','situacao_desc','cnae_descricao',
    'score_final','if_anomalia','dbscan_outlier',
    'anos_operacao','qtd_socios','qtd_filiais',
    'faixa_capital','max_empresas_por_socio'
]].copy()
resumo.columns = [
    'ID','Situação','Segmento','Score Final',
    'IF Anômalo','DBSCAN Outlier',
    'Idade (anos)','Sócios','Filiais',
    'Faixa Capital','Max Emp/Sócio'
]
resumo['IF Anômalo'] = resumo['IF Anômalo'].map({1:'✅ Sim', 0:'❌ Não'})
resumo['DBSCAN Outlier'] = resumo['DBSCAN Outlier'].map({1:'✅ Sim', 0:'❌ Não'})
resumo = resumo.set_index('ID')
display(resumo)

In [ ]:
# Análise detalhada caso a caso
print('=' * 70)
print('  ANÁLISE DETALHADA — 8 CASOS CRÍTICOS')
print('  (valores de capital apresentados em faixas — privacidade LGPD)')
print('=' * 70)

for i, (_, row) in enumerate(criticos.iterrows(), 1):
    ambos = row['if_anomalia']==1 and row['dbscan_outlier']==1
    print(f'\n  CASO {i}: {row["id_empresa"]}')
    print(f'  Segmento:    {row["cnae_descricao"]}')
    print(f'  Situação:    {row["situacao_desc"]}')
    print(f'  Score final: {row["score_final"]:.3f} | Rank: #{int(row["ranking"])}')
    print(f'  Detectado por ambos os modelos: {"✅ Sim" if ambos else "⚠️ Apenas um"}')
    print(f'  ── Perfil cadastral:')
    print(f'     Idade:           {row["anos_operacao"]:.1f} anos')
    print(f'     Capital social:  {row["faixa_capital"]}')
    print(f'     Sócios:          {int(row["qtd_socios"])} ({"sem sócios" if row["sem_socios"]==1 else "com sócios"})')
    print(f'     Filiais:         {int(row["qtd_filiais"])}')
    print(f'     Max emp/sócio:   {row["max_empresas_por_socio"]:.0f} empresas')
    print(f'     Taxa inapta mun: {row["taxa_inapta_municipio"]:.1%}')
    
    # Interpretação fiscal
    alertas = []
    if row['capital_social'] > 1e6 and row['anos_operacao'] < 2:
        alertas.append('Capital incompatível com tempo de operação')
    if row['sem_socios'] == 1:
        alertas.append('Ausência de sócios — responsabilidade legal indefinida')
    if row['max_empresas_por_socio'] > 10:
        alertas.append(f'Sócio com {row["max_empresas_por_socio"]:.0f} empresas — possível laranja')
    if row['taxa_inapta_municipio'] > 0.20:
        alertas.append(f'Município com {row["taxa_inapta_municipio"]:.0%} de empresas inaptas — risco regional')
    if row['qtd_filiais'] > 3 and row['anos_operacao'] < 2:
        alertas.append('Muitas filiais para empresa recém-aberta')
    if row['inapta'] == 1 and row['anos_operacao'] > 10:
        alertas.append('Empresa antiga em situação inapta — possível abandono ou fraude histórica')
    
    if alertas:
        print(f'  ── Alertas fiscais:')
        for a in alertas:
            print(f'     ⚠️  {a}')

## 4. Análise dos Clusters Suspeitos (DBSCAN)

In [ ]:
# Perfil dos clusters mais suspeitos
clusters_df = df[df['dbscan_cluster'] >= 0].copy()

perfil_clusters = clusters_df.groupby('dbscan_cluster').agg(
    qtd=('id_empresa', 'count'),
    score_medio=('score_final', 'mean'),
    pct_inaptas=('inapta', 'mean'),
    pct_sem_socios=('sem_socios', 'mean'),
    capital_mediano=('capital_social', 'median'),
    anos_medio=('anos_operacao', 'mean'),
    socios_medio=('qtd_socios', 'mean'),
    pct_if_anomalo=('if_anomalia', 'mean'),
).round(3).sort_values('score_medio', ascending=False)

top_clusters = perfil_clusters.head(8)
print('Top 8 clusters por score médio:')
display(top_clusters)

In [ ]:
# Visualização dos clusters suspeitos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

top8_ids = top_clusters.index.tolist()
top8_df  = clusters_df[clusters_df['dbscan_cluster'].isin(top8_ids)].copy()
top8_df['cluster_label'] = 'C' + top8_df['dbscan_cluster'].astype(str)

# Score médio por cluster
scores_c = top_clusters['score_medio'].sort_values()
labels_c = [f'C{i}' for i in scores_c.index]
cores_c  = [AZUL['escuro'] if v > scores_c.mean() else AZUL['claro']
             for v in scores_c.values]
axes[0,0].barh(range(len(scores_c)), scores_c.values,
                color=cores_c, alpha=0.85, edgecolor='white', linewidth=1.5)
axes[0,0].set_yticks(range(len(scores_c)))
axes[0,0].set_yticklabels(labels_c)
axes[0,0].axvline(scores_c.mean(), color=VERMELHO, linestyle='--',
                   linewidth=2, label=f'Média: {scores_c.mean():.3f}')
axes[0,0].set_title('Score Médio por Cluster')
axes[0,0].set_xlabel('Score Final médio')
axes[0,0].legend()

# % inaptas por cluster
pct_i = top_clusters['pct_inaptas'].sort_values() * 100
cores_i = [AZUL['escuro'] if v > 50 else AZUL['claro'] for v in pct_i.values]
axes[0,1].barh(range(len(pct_i)), pct_i.values,
                color=cores_i, alpha=0.85, edgecolor='white', linewidth=1.5)
axes[0,1].set_yticks(range(len(pct_i)))
axes[0,1].set_yticklabels([f'C{i}' for i in pct_i.index])
axes[0,1].axvline(pct_i.mean(), color=VERMELHO, linestyle='--',
                   linewidth=2, label=f'Média: {pct_i.mean():.1f}%')
for i, val in enumerate(pct_i.values):
    axes[0,1].text(val+1, i, f'{val:.0f}%', va='center', fontsize=9,
                   fontweight='bold', color=AZUL['escuro'])
axes[0,1].set_title('% de Empresas Inaptas por Cluster')
axes[0,1].set_xlabel('% inaptas')
axes[0,1].legend()

# Tamanho dos clusters
qtd_c = top_clusters['qtd'].sort_values(ascending=False)
axes[1,0].bar(range(len(qtd_c)), qtd_c.values,
               color=AZUL['claro'], alpha=0.85, edgecolor='white', linewidth=1.5)
axes[1,0].set_xticks(range(len(qtd_c)))
axes[1,0].set_xticklabels([f'C{i}' for i in qtd_c.index])
for i, val in enumerate(qtd_c.values):
    axes[1,0].text(i, val+0.5, f'{val:,}', ha='center',
                   fontweight='bold', color=AZUL['escuro'])
axes[1,0].set_title('Tamanho dos Clusters (número de empresas)')
axes[1,0].set_ylabel('Empresas')

# % sem sócios por cluster
pct_ss = top_clusters['pct_sem_socios'].sort_values() * 100
axes[1,1].barh(range(len(pct_ss)), pct_ss.values,
                color=AZUL['medio'], alpha=0.85, edgecolor='white', linewidth=1.5)
axes[1,1].set_yticks(range(len(pct_ss)))
axes[1,1].set_yticklabels([f'C{i}' for i in pct_ss.index])
axes[1,1].axvline(pct_ss.mean(), color=VERMELHO, linestyle='--',
                   linewidth=2, label=f'Média: {pct_ss.mean():.1f}%')
for i, val in enumerate(pct_ss.values):
    axes[1,1].text(val+1, i, f'{val:.0f}%', va='center', fontsize=9,
                   fontweight='bold', color=AZUL['escuro'])
axes[1,1].set_title('% Empresas Sem Sócios por Cluster')
axes[1,1].set_xlabel('% sem sócios')
axes[1,1].legend()

plt.suptitle('Perfil dos Clusters Mais Suspeitos — DBSCAN\nCadeia de Combustíveis RJ',
             fontsize=13, fontweight='bold', color=AZUL['escuro'])
plt.tight_layout()
plt.savefig(OUT_DIR/'12_perfil_clusters_suspeitos.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

## 5. Padrões de Anomalia Identificados

In [ ]:
# Classificar empresas por padrão de anomalia
alto_risco = df[df['classe_final'].isin(['Alto','Crítico'])].copy()

# Definir padrões
alto_risco['padrao'] = 'Outros'

# Padrão 1: Capital incompatível (ativo + capital alto + empresa nova)
mask1 = ((alto_risco['capital_social'] > 1e6) &
          (alto_risco['anos_operacao'] < 2) &
          (alto_risco['situacao_desc'] == 'Ativa'))
alto_risco.loc[mask1, 'padrao'] = 'P1: Capital incompatível'

# Padrão 2: Empresa fantasma (sem sócios + inapta)
mask2 = ((alto_risco['sem_socios'] == 1) &
          (alto_risco['inapta'] == 1))
alto_risco.loc[mask2, 'padrao'] = 'P2: Empresa fantasma'

# Padrão 3: Laranja societário (sócio com muitas empresas)
mask3 = (alto_risco['max_empresas_por_socio'] > 10)
alto_risco.loc[mask3 & (alto_risco['padrao']=='Outros'), 'padrao'] = 'P3: Laranja societário'

# Padrão 4: Abandono fiscal (antiga + inapta)
mask4 = ((alto_risco['anos_operacao'] > 10) &
          (alto_risco['inapta'] == 1))
alto_risco.loc[mask4 & (alto_risco['padrao']=='Outros'), 'padrao'] = 'P4: Abandono fiscal'

# Padrão 5: Ativa suspeita (ativa + IF anômalo + sem sócios)
mask5 = ((alto_risco['situacao_desc'] == 'Ativa') &
          (alto_risco['if_anomalia'] == 1) &
          (alto_risco['sem_socios'] == 1))
alto_risco.loc[mask5 & (alto_risco['padrao']=='Outros'), 'padrao'] = 'P5: Ativa suspeita'

dist_padroes = alto_risco['padrao'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
cores_p = [AZUL['escuro'], AZUL['medio'], AZUL['claro'],
            AZUL['palido'], AZUL['cinza'], '#85929e']
bars = ax.barh(range(len(dist_padroes)), dist_padroes.values,
                color=cores_p[:len(dist_padroes)], alpha=0.85,
                edgecolor='white', linewidth=1.5)
ax.set_yticks(range(len(dist_padroes)))
ax.set_yticklabels(dist_padroes.index, fontsize=10)
for i, val in enumerate(dist_padroes.values):
    ax.text(val+0.5, i, f'{val:,} ({val/len(alto_risco)*100:.1f}%)',
             va='center', fontweight='bold', color=AZUL['escuro'])
ax.set_title('Distribuição por Padrão de Anomalia\n(empresas de risco Alto + Crítico)',
              fontsize=12, fontweight='bold', color=AZUL['escuro'])
ax.set_xlabel('Número de empresas')
plt.tight_layout()
plt.savefig(OUT_DIR/'13_padroes_anomalia.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

print(f'\nTotal empresas Alto+Crítico: {len(alto_risco):,}')
print(f'Distribuição por padrão:')
for padrao, qtd in dist_padroes.items():
    print(f'  {padrao}: {qtd:,} ({qtd/len(alto_risco)*100:.1f}%)')

## 6. Implicações para a Fiscalização Fiscal

In [ ]:
# Ganho de eficiência do modelo vs fiscalização tradicional
total = len(df)
criticos_n = len(df[df['classe_final']=='Crítico'])
alto_n     = len(df[df['classe_final']=='Alto'])
prioritarios = criticos_n + alto_n

# Premissas de fiscalização
# Um auditor consegue fiscalizar ~50 empresas/ano em profundidade
auditores = 3
capacidade_anual = auditores * 50

print('=' * 65)
print('  ANÁLISE DE EFICIÊNCIA — MODELO VS MÉTODO TRADICIONAL')
print('=' * 65)
print(f'\n  Universo total de empresas:        {total:>8,}')
print(f'  Empresas prioritárias (Alto+Crítico):{prioritarios:>7,} ({prioritarios/total*100:.1f}%)')
print(f'\n  MÉTODO TRADICIONAL (sorteio aleatório):')
print(f'  Probabilidade de sortear suspeita: {prioritarios/total*100:.1f}%')
print(f'  Em {capacidade_anual} fiscalizações/ano:')
print(f'  Suspeitas encontradas (esperado):  {capacidade_anual * prioritarios/total:.0f}')
print(f'\n  MÉTODO COM MODELO (score combinado):')
print(f'  Fiscalização focada nas {prioritarios} empresas prioritárias')
print(f'  Taxa de acerto esperada:           ~100% no grupo prioritário')
print(f'  Suspeitas encontradas:             {min(prioritarios, capacidade_anual)}')
print(f'\n  GANHO DE EFICIÊNCIA:')
ganho = min(prioritarios, capacidade_anual) / max(1, capacidade_anual * prioritarios/total)
print(f'  Multiplicador de eficiência:       ~{ganho:.0f}x')
print(f'  Redução do universo a fiscalizar:  {(1 - prioritarios/total)*100:.1f}%')
print('=' * 65)

In [ ]:
# Visualização do ganho de eficiência
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pirâmide de priorização
categorias = ['Crítico\n(8)', 'Alto\n(142)', 'Médio\n(294)', 'Baixo\n(1.076)']
valores    = [8, 142, 294, 1076]
cores_pir  = [AZUL['escuro'], AZUL['medio'], AZUL['claro'], AZUL['palido']]
bars = axes[0].barh(range(4), valores, color=cores_pir,
                     edgecolor='white', linewidth=1.5, alpha=0.9)
axes[0].set_yticks(range(4))
axes[0].set_yticklabels(categorias[::-1][::-1], fontsize=10)
axes[0].invert_yaxis()
for bar, val in zip(bars, valores):
    axes[0].text(val+5, bar.get_y()+bar.get_height()/2,
                  f'{val:,} ({val/total*100:.1f}%)',
                  va='center', fontweight='bold', color=AZUL['escuro'])
axes[0].axvline(capacidade_anual, color=VERMELHO, linestyle='--',
                 linewidth=2, label=f'Capacidade anual ({capacidade_anual} auditores)')
axes[0].set_title('Pirâmide de Priorização Fiscal\n(empresas por classe de risco)')
axes[0].set_xlabel('Número de empresas')
axes[0].legend(fontsize=9)

# Comparativo eficiência
metodos = ['Sorteio\nAleatório', 'Modelo\nML']
suspeitas_encontradas = [
    capacidade_anual * prioritarios/total,
    min(prioritarios, capacidade_anual)
]
bars2 = axes[1].bar(metodos, suspeitas_encontradas,
                     color=[AZUL['palido'], AZUL['escuro']],
                     edgecolor='white', linewidth=1.5, alpha=0.9, width=0.5)
for bar, val in zip(bars2, suspeitas_encontradas):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                  f'{val:.0f} empresas', ha='center',
                  fontweight='bold', color=AZUL['escuro'])
axes[1].set_title(f'Empresas Suspeitas Encontradas\npor {capacidade_anual} Fiscalizações/ano')
axes[1].set_ylabel('Empresas suspeitas encontradas')
axes[1].set_ylim(0, max(suspeitas_encontradas)*1.3)

plt.suptitle('Eficiência do Modelo para Fiscalização Tributária\nCadeia de Combustíveis RJ',
             fontsize=13, fontweight='bold', color=AZUL['escuro'])
plt.tight_layout()
plt.savefig(OUT_DIR/'14_eficiencia_fiscalizacao.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

## 7. Protocolo de Priorização para Auditores

In [ ]:
print('=' * 70)
print('  PROTOCOLO DE PRIORIZAÇÃO FISCAL BASEADO EM ML')
print('  Cadeia de Combustíveis RJ | Nov/2025')
print('=' * 70)

print('''
  NÍVEL 1 — AÇÃO IMEDIATA (Score ≥ 0.70 | 8 empresas)
  ─────────────────────────────────────────────────────
  • Iniciar procedimento de fiscalização aprofundada
  • Solicitar SPED Fiscal e EFD-ICMS dos últimos 5 anos
  • Verificar NF-e emitidas e recebidas vs. movimento declarado
  • Cruzar com dados de fornecedores e clientes (análise de rede)
  • Prioridade: empresas ativas com capital incompatível

  NÍVEL 2 — MONITORAMENTO ATIVO (Score 0.50-0.70 | 142 empresas)
  ────────────────────────────────────────────────────────────────
  • Incluir na malha fina de ICMS
  • Verificar regularidade das obrigações acessórias
  • Cruzar declarações com movimento bancário
  • Agendar diligência fiscal no prazo de 90 dias
  • Atenção especial: sócios com múltiplas empresas

  NÍVEL 3 — ACOMPANHAMENTO (Score 0.30-0.50 | 294 empresas)
  ───────────────────────────────────────────────────────────
  • Monitoramento por cruzamento eletrônico de dados
  • Verificar regularidade cadastral anualmente
  • Alertar para obrigações de atualização cadastral
  • Reavaliar score a cada novo período de referência

  NÍVEL 4 — SITUAÇÃO REGULAR (Score < 0.30 | 1.076 empresas)
  ─────────────────────────────────────────────────────────────
  • Manter monitoramento padrão
  • Reavaliar periodicamente com nova base de dados
  • Não requer ação fiscal imediata
''')

print('  PADRÕES DE FRAUDE IDENTIFICADOS:')
print('''
  P1 — Capital incompatível com operação
       Indicativo: subfaturamento ou superavaliação patrimonial
       Legislação: Art. 72 RICMS-RJ | Lei 5.172/66 Art. 149

  P2 — Empresa fantasma (sem sócios + inapta)
       Indicativo: emissão de NF-e frias para transferência de créditos
       Legislação: Art. 11 Lei 8.137/90 | Súmula CARF 14

  P3 — Laranja societário (sócio com >10 empresas)
       Indicativo: estrutura de blindagem patrimonial ou ocultação
       Legislação: Art. 124 CTN | Art. 135 CTN (responsabilidade)

  P4 — Abandono fiscal (antiga + inapta)
       Indicativo: créditos acumulados não compensados ou débitos ocultos
       Legislação: Art. 60 Lei 9.430/96 | Art. 173 CTN

  P5 — Ativa suspeita (ativa + IF anômalo + sem sócios)
       Indicativo: empresa operando sem responsabilidade legal definida
       Legislação: Art. 1.052 CC | Art. 134 CTN
''')
print('=' * 70)

In [ ]:
# Exportar lista priorizada (anonimizada) para uso do auditor
lista_auditoria = df[df['classe_final'].isin(['Crítico','Alto'])][
    ['id_empresa','situacao_desc','cnae_descricao',
     'score_final','classe_final','ranking',
     'if_anomalia','dbscan_outlier','score_manual_norm']
].sort_values('score_final', ascending=False).copy()

# Adicionar padrão identificado
lista_auditoria = lista_auditoria.merge(
    alto_risco[['id_empresa','padrao','anos_operacao',
                'qtd_socios','qtd_filiais','sem_socios',
                'max_empresas_por_socio','taxa_inapta_municipio']],
    on='id_empresa', how='left'
)

caminho = Path('../outputs/tabelas/lista_prioridade_auditoria.csv')
lista_auditoria.to_csv(caminho, index=False, encoding='utf-8-sig')
print(f'✅ Lista de prioridade salva: {caminho}')
print(f'   {len(lista_auditoria):,} empresas para fiscalização prioritária')
display(lista_auditoria.head(10))